In [ ]:
!pip install gdown pandas numpy scipy

In [5]:
gid_map = {
    'profiling_agg': '1092807689'
}

sheet_id = "1e_lKct9ovnYByYkGpFCKrMpKQs5TQdKhTld9tDU1JcQ"

In [38]:
import gdown
import pandas as pd
import numpy as np
from scipy import stats


def get_pandas_from_gid(gid_name):
    # Replace with your actual Google Sheet ID
    # Optionally: specify the sheet/tab number (gid=0 for the first sheet)
    gid = gid_map[gid_name]
    
    # Construct export URL
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    
    # Set desired output filename
    output = f"{gid_name}.csv"
    
    # Download CSV using gdown
    gdown.download(url, output, quiet=False)
    original_df = pd.read_csv(output)
    return original_df

In [39]:
df = get_pandas_from_gid('profiling_agg')
df

/home/diogoocv/miniconda3/lib/python3.13/site-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(
Downloading...
From: https://docs.google.com/spreadsheets/d/1e_lKct9ovnYByYkGpFCKrMpKQs5TQdKhTld9tDU1JcQ/export?format=csv&gid=1092807689
To: /home/diogoocv/IdeaProjects/fractal/plots/CCPE_Plots/profiling_agg.csv
3.99kB [00:00, 2.65MB/s]


,graph,obj_function,metaheuristic,AVERAGE of obj_function_perc,STDEV of obj_function_perc,COUNT of obj_function_perc,confidence_interval
0,amazon,conductance,ils,40.83,12.75,30,4.76
1,amazon,conductance,ts1,16.71,5.95,30,2.22
2,amazon,conductance,vns,23.34,14.16,30,5.29
3,amazon,degreeentropy,ils,67.44,9.12,30,3.41
4,amazon,degreeentropy,ts1,55.76,5.40,30,2.02
...,...,...,...,...,...,...,...
76,youtube,labelentropy,ts1,69.08,5.73,30,2.14
77,youtube,labelentropy,vns,72.67,5.86,30,2.19
78,youtube,triangledensestsubgraph,ils,89.44,5.31,30,1.98
79,youtube,triangledensestsubgraph,ts1,89.82,4.18,30,1.56


In [44]:
# Global replacements
df['obj_function'] = df['obj_function'].replace({
    'conductance': 'CO',
    'triangledensestsubgraph': 'TDS',
    'densesubgraph': 'DS',
    'degreeentropy': 'DE',
    'labelentropy': 'LE'
})

df['graph'] = df['graph'].replace({
    'citeseer': 'Citeseer',
    'amazon': 'Amazon',
    'dblp': 'DBLP',
    'patents': 'Patents',
    'livejournal': 'LiveJournal',
    'youtube': 'Youtube',
})

df['metaheuristic'] = df['metaheuristic'].replace({
    'vns': 'VNS',
    'ils': 'ILS',
    'ts1': 'TS',
})

# Define the columns mapping
new_names = {
    'obj_function': 'Scoring Function',
    'graph': 'Graph',
    'AVERAGE of obj_function_perc': 'Time in Scoring Function (\\%)',
    'confidence_interval': 'Confidence Interval'
}

# Define Sorting Orders
graph_order = ['Citeseer', 'Amazon', 'DBLP', 'Patents', 'LiveJournal', 'Youtube']
scoring_function_order = ['DS', 'CO', 'DE', 'LE', 'TDS']

# Loop through each metaheuristic to create separate tables
target_metaheuristics = ['VNS', 'ILS', 'TS']
for mh in target_metaheuristics:
    print(f"\n% --- Table {mh} --- ")

    # Filter data for the current metaheuristic
    df_subset = df[df['metaheuristic'] == mh].copy()

    # Create the Combined String "Mean \pm CI"
    # Round to 2 decimals first to ensure clean output
    mean_str = df_subset['AVERAGE of obj_function_perc'].apply(lambda x: f"{x:.2f}")
    ci_str = df_subset['confidence_interval'].apply(lambda x: f"{x:.2f}")

    df_subset['formatted_cell'] = mean_str + r'$\pm$' + ci_str

    # Turns Graphs into Columns
    pivot_df = df_subset.pivot(
        index='obj_function',
        columns='graph',
        values='formatted_cell'
    )

    # Enforce exact order of Rows and Columns
    pivot_df = pivot_df.reindex(index=scoring_function_order, columns=graph_order)

    # Fill missing values (like LE on Amazon) with '-'
    pivot_df = pivot_df.fillna('-')

    pivot_df.index.name = None

    # Convert the inner data to a LaTeX string body
    latex_body = pivot_df.to_latex(
        column_format="l rrrrrr",
        escape=False,
        header=False,
        index=True
    )

    # Clean up the body to remove the top/bottom rules pandas adds
    # We just want the data rows
    latex_body_content = "\n".join(latex_body.splitlines()[2:-2])

    # --- CONSTRUCT YOUR EXACT TEMPLATE ---
    final_latex = f"""
\\begin{{table}}[!htb]
\\caption{{Impact of Scoring Functions using {mh}.}}
\\label{{tab:scoring-functions-{mh}}}
\\footnotesize
\\centering
\\begin{{tabular}}{{l rrrrrr}}
\\toprule
 & Citeseer & Amazon & DBLP & Patents & LiveJournal & Youtube \\\\
\\midrule
{latex_body_content}
\\bottomrule
\\multicolumn{{7}}{{c}}{{\\scriptsize Missing entries (-) are due to Label Entropy (LE) not applicable for unlabeled graphs}}
\\end{{tabular}}
\\end{{table}}
"""
    print(final_latex)


% --- Table VNS --- 

\begin{table}[!htb]
\caption{Impact of Scoring Functions using VNS.}
\label{tab:scoring-functions-VNS}
\footnotesize
\centering
\begin{tabular}{l rrrrrr}
\toprule
 & Citeseer & Amazon & DBLP & Patents & LiveJournal & Youtube \\
\midrule
\midrule
DS & 0.65$\pm$0.05 & 0.59$\pm$0.04 & 0.28$\pm$0.05 & 0.57$\pm$0.05 & 0.19$\pm$0.03 & 0.43$\pm$0.04 \\
CO & 17.33$\pm$4.13 & 23.34$\pm$5.29 & 4.59$\pm$0.19 & 8.20$\pm$0.28 & 4.28$\pm$0.17 & 8.33$\pm$2.82 \\
DE & 49.84$\pm$5.83 & 68.07$\pm$2.67 & 53.86$\pm$1.93 & 44.41$\pm$9.86 & 49.75$\pm$3.18 & 64.05$\pm$2.00 \\
LE & 55.59$\pm$2.06 & - & - & 62.44$\pm$1.29 & - & 72.67$\pm$2.19 \\
TDS & 71.53$\pm$1.91 & 77.51$\pm$1.83 & 85.33$\pm$1.43 & 89.05$\pm$1.52 & 90.51$\pm$1.80 & 90.10$\pm$1.89 \\
\bottomrule
\multicolumn{7}{c}{\scriptsize Missing entries (-) are due to Label Entropy (LE) not applicable for unlabeled graphs}
\end{tabular}
\end{table}


% --- Table ILS --- 

\begin{table}[!htb]
\caption{Impact of Scoring Functions us